# Converting unknown GSDR data to PDEBench Format

Given that the data structures of both are very similar, the conversion is relatively straight forward.

1) Scale the ranges of GSDR data accordingly

2) Appropriate spatio-temporal scaling of data

3) Tranpose data order accordingly to match PDEBench DR's convention.

Observation: DPOT does not seem to be able to generalize well to unknown GSDR data.

In [1]:
from DPOT.models.dpot import DPOTNet
import torch

model = DPOTNet(img_size=128, patch_size=8, mixing_type='afno', in_channels=4, in_timesteps=10, out_timesteps=1, out_channels=4, normalize=False, embed_dim=512, modes=32, depth=4, n_blocks=4, mlp_ratio=1, out_layer_dim=32, n_cls=12)
model.load_state_dict(torch.load('./DPOT/model_Ti.pth')['model'])
print(model)

DPOTNet(
  (pos_embed): tensor((1, 512, 16, 16), requires_grad=True)
  
  (act): GELU(approximate='none')
  (patch_embed): PatchEmbed(
    (act): GELU(approximate='none')
    (proj): Sequential(
      (0): Conv2d(7, 35, kernel_size=(8, 8), stride=(8, 8))
      (1): GELU(approximate='none')
      (2): Conv2d(35, 512, kernel_size=(1, 1), stride=(1, 1))
    )
  )
  (blocks): ModuleList(
    (0-3): 4 x Block(
      (norm1): GroupNorm(8, 512, eps=1e-05, affine=True)
      (act): GELU(approximate='none')
      (filter): AFNO2D(
        (act): GELU(approximate='none')
      )
      (norm2): GroupNorm(8, 512, eps=1e-05, affine=True)
      (mlp): Sequential(
        (0): Conv2d(512, 512, kernel_size=(1, 1), stride=(1, 1))
        (1): GELU(approximate='none')
        (2): Conv2d(512, 512, kernel_size=(1, 1), stride=(1, 1))
      )
    )
  )
  (cls_head): Sequential(
    (0): Linear(in_features=512, out_features=512, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=512, o

In [11]:
import h5py
import numpy as np

pdb_dr = '/Volumes/T7/PDEBench/2D/diffusion-reaction/dr_sliced_0_99.h5'

with h5py.File(pdb_dr, 'r') as f:
    print(f['0000'].keys())
    print(f['0000']['data'])

    ds = f['0000']['data']

    num_channels = 2

    for i in range(num_channels):
        channel_data = ds[..., i]
        c_min = np.min(channel_data)
        c_max = np.max(channel_data)

        print(f"Channel {i}:")
        print(f"  Min: {c_min}")
        print(f"  Max: {c_max}")

<KeysViewHDF5 ['data', 'grid']>
<HDF5 dataset "data": shape (101, 128, 128, 2), type "<f4">
Channel 0:
  Min: -3.5104098320007324
  Max: 4.241771697998047
Channel 1:
  Min: -4.24711799621582
  Max: 3.8653523921966553


In [7]:
import scipy.io

file_path = '/Volumes/T7/New_Data/2d_gs_rd/data/2DRD_2x3001x100x100_[dt=05].mat'

data = scipy.io.loadmat(file_path)

print(data.keys())

uv_data = data['uv']
print(uv_data.shape)
print(uv_data.dtype)

# 2 channels
# 3001 timesteps
# 100 by 100

dict_keys(['__header__', '__version__', '__globals__', 'uv'])
(2, 3001, 100, 100)
float64


In [5]:
import h5py
import numpy as np
import scipy.io
from scipy.interpolate import interp1d
from scipy.ndimage import zoom

file_path = '/Volumes/T7/New_Data/2d_gs_rd/data/2DRD_2x3001x100x100_[dt=05].mat'
data = scipy.io.loadmat(file_path)
uv_data = data['uv']

PDB_MIN = -4.5
PDB_MAX = 4.5

#Temporal interpolation
c, t_old, u, v = uv_data.shape
t_new = (t_old - 1) * 10 + 1

old_ticks = np.linspace(0, 1, t_old)
new_ticks = np.linspace(0, 1, t_new)

f_interp = interp1d(old_ticks, uv_data, axis=1, kind='linear')
expanded_data = f_interp(new_ticks).astype(np.float32)

# Scaling to PDEBench DR's range
for i in range(c):
    channel_min = expanded_data[i].min()
    channel_max = expanded_data[i].max()

    expanded_data[i] = (expanded_data[i] - channel_min) / (channel_max - channel_min)
    expanded_data[i] = expanded_data[i] * (PDB_MAX - PDB_MIN) + PDB_MIN

# Format to PDEBench convention: (t, x, y, c)
pde_format_temp = expanded_data.transpose(1, 2, 3, 0)

# Spatial upsampling
zoom_factors = (1, 128/100, 128/100, 1)
pde_final_data = zoom(pde_format_temp, zoom_factors, order=1)

# Save to hdf5
h5_file_path = '/Volumes/T7/New_Data/2d_gs_rd/data/2DRD.h5'

with h5py.File(h5_file_path, 'w') as f:
    group = f.create_group('0000')
    group.create_dataset(
        'data',
        data=pde_final_data,
        dtype='f4',
    )

print(f"File saved: {h5_file_path}")
print(f"Final shape: {pde_final_data.shape}")
print(f"New Min: {pde_final_data.min():.4f} | New Max: {pde_final_data.max():.4f}")

File saved: /Volumes/T7/New_Data/2d_gs_rd/data/2DRD.h5
Final shape: (30001, 128, 128, 2)
New Min: -4.5000 | New Max: 4.5000


In [6]:
import h5py

file = '/Volumes/T7/New_Data/2d_gs_rd/data/2DRD.h5'

with h5py.File(file, 'r') as f:
    print(f.keys())
    print(f['0000'])
    print(f['0000']['data'])

<KeysViewHDF5 ['0000']>
<HDF5 group "/0000" (1 members)>
<HDF5 dataset "data": shape (30001, 128, 128, 2), type "<f4">


In [7]:
from DPOT.data_generation.preprocess import process_dr_pdebench

process_dr_pdebench(
    path = '/Volumes/T7/New_Data/2d_gs_rd/data/2DRD.h5',
    save_name = '/Volumes/T7/New_Data/2d_gs_rd/DPOT_Processed',
    n_train = 0,
    n_test = 1
)

path created
(1, 128, 128, 30001, 2)
train ids []
test ids [0]
task @ 0 saved, shape (128, 128, 30001, 2)
file saved


In [23]:
!python DPOT/evaluate.py \
--model DPOT \
--dataset dr_pdb \
--train_paths dr_pdb \
--test_paths dr_pdb \
--resume_path DPOT/model_S.pth \
--n_channels 4 \
--T_in 10 \
--res 128 \
--batch_size 1 \
--ntest_list 1 \
--gpu cpu

python(86023) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Current working directory: /Users/sky/Git/DPOT
args Namespace(model='DPOT', dataset='dr_pdb', train_paths=['dr_pdb'], test_paths=['dr_pdb'], resume_path='DPOT/model_S.pth', ntrain_list=[100], ntest_list=[1], data_weights=[1], use_writer=False, res=128, noise_scale=0.0, width=1024, n_layers=6, act='gelu', max_nodes=-1, modes=20, use_ln=0, normalize=0, patch_size=8, n_blocks=8, mlp_ratio=1, out_layer_dim=32, batch_size=1, epochs=500, lr=0.001, opt='adam', beta1=0.9, beta2=0.999, lr_method='step', grad_clip=10000.0, step_size=100, step_gamma=0.5, warmup_epochs=50, sub=1, T_in=10, T_ar=1, T_bundle=1, gpu='cpu', comment='', log_path='', n_channels=4, n_class=12)
Train num [100] test num [1]
Loading models and fine tune from DPOT/model_S.pth
Using step learning rate schedule
DPOTNet(
  (pos_embed): tensor((1, 1024, 16, 16), requires_grad=True)
  
  (act): GELU(approximate='none')
  (patch_embed): PatchEmbed(
    (act): GELU(approximate='none')
    (proj): Sequential(
      (0): Conv2d(7, 35,